# Student-Teacher Anomaly Detection

This notebook trains a Student-Teacher feature-distillation branch on `train/good/` and evaluates pixel-level anomaly maps. It integrates with the existing project configuration and can be ensembled with PatchCore maps later.

In [ ]:
import os
from pathlib import Path
import sys

# Change directory to project root if running from notebooks/
if Path.cwd().name == "notebooks":
    os.chdir("..")

sys.path.append('.')

# Imports and config
import matplotlib.pyplot as plt
from adl_lib.config import IMAGE_SIZE, DEVICE, PATH, CLASS_NAME, SEED
from adl_lib.student_teacher import (
    set_seed,
    train_student_teacher,
    make_transforms,
    build_student_teacher_splits,
    validate_student_teacher_splits,
    PixelValDataset,
    show_student_teacher_predictions,
)

In [ ]:
# Hyperparameters
T_BACKBONE = "wide_resnet50_2"
S_BACKBONE = "resnet18"
ST_OUT_INDICES = (1, 2, 3)
ST_N_STUDENTS = 3
ST_STUDENT_PRETRAINED = False
ST_EPOCHS = 25
ST_LR = 3e-4
ST_WEIGHT_DECAY = 1e-4
ST_BATCH_SIZE = 16
ST_TOPK_PERCENT = 1.0

ST_SAVE_DIR = Path("./student_teacher_models")
ST_SAVE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Build and validate dataset splits before training
train_good_paths, val_items = build_student_teacher_splits(
    path=PATH,
    class_name=CLASS_NAME,
    image_size=IMAGE_SIZE,
    seed=SEED,
)

# Validate: check that all anomalies have ground truth masks
# validation_stats = validate_student_teacher_splits(train_good_paths, val_items)

In [ ]:
# Train for a single class (example).
set_seed(SEED)
model_st, history_st, save_path_st = train_student_teacher(
    path=PATH,
    class_name=CLASS_NAME,
    image_size=IMAGE_SIZE,
    backbone=T_BACKBONE,
    student_backbone=S_BACKBONE,
    out_indices=ST_OUT_INDICES,
    n_students=ST_N_STUDENTS,
    student_pretrained=ST_STUDENT_PRETRAINED,
    epochs=ST_EPOCHS,
    batch_size=ST_BATCH_SIZE,
    lr=ST_LR,
    weight_decay=ST_WEIGHT_DECAY,
    seed=SEED,
    device=DEVICE,
    topk_percent=ST_TOPK_PERCENT,
)

In [ ]:
# Visualize a few validation predictions
_, eval_tf = make_transforms(IMAGE_SIZE)
_, val_items = build_student_teacher_splits(path=PATH, class_name=CLASS_NAME, image_size=IMAGE_SIZE, seed=SEED)
val_ds_st = PixelValDataset(val_items, transform=eval_tf, image_size=IMAGE_SIZE)
show_student_teacher_predictions(model_st, val_ds_st, n=6, image_size=IMAGE_SIZE, device=DEVICE)

In [ ]:
# Final submission generation for all classes
from pathlib import Path
import gc
import time
import zipfile

import pandas as pd
import torch
from torch.utils.data import DataLoader

from adl_lib.config import NUM_WORKERS, PIN_MEMORY
from adl_lib.student_teacher import (
    ImagePathDataset,
    StudentTeacherAD,
    compute_student_teacher_maps,
    list_images,
    make_transforms,
    set_seed,
    train_student_teacher,
)
from adl_lib.utils import float_matrix_to_q8rle, normalize_map


def load_student_teacher_checkpoint(save_path: Path, device: str):
    checkpoint = torch.load(save_path, map_location=device)
    model_state = checkpoint["model"]

    model = StudentTeacherAD(
        teacher_backbone=model_state["teacher_backbone"],
        student_backbone=model_state["student_backbone"],
        out_indices=tuple(model_state["out_indices"]),
        n_students=int(model_state["n_students"]),
        device=device,
    ).to(device)

    for student, student_state in zip(model.students, model_state["students"]):
        student.load_state_dict(student_state)

    for projector, projector_state in zip(
        getattr(model, "student_projectors", []),
        model_state.get("student_projectors", []),
    ):
        projector.load_state_dict(projector_state)

    if model_state.get("score_stats") is not None:
        model.set_score_stats(model_state["score_stats"])

    return model


classes = sorted(
    [p.name for p in Path(PATH).iterdir() if p.is_dir() and p.name.startswith("class_")]
)
print("Detected classes:", classes)

train_tf, eval_tf = make_transforms(IMAGE_SIZE)
submission_rows = []
start_all = time.time()

for cls in classes:
    t0 = time.time()
    print(f"\n{'=' * 60}")
    print(f"Processing {cls}...")
    print('=' * 60)

    save_path = ST_SAVE_DIR / f"student_teacher_{cls}_T-{T_BACKBONE}_S-{S_BACKBONE}.pt"

    if save_path.exists():
        print(f"Loading existing checkpoint: {save_path}")
        model_st = load_student_teacher_checkpoint(save_path, DEVICE)
    else:
        print("Training student-teacher model from scratch...")
        set_seed(SEED)
        model_st, _, save_path = train_student_teacher(
            path=PATH,
            class_name=cls,
            image_size=IMAGE_SIZE,
            backbone=T_BACKBONE,
            student_backbone=S_BACKBONE,
            out_indices=ST_OUT_INDICES,
            n_students=ST_N_STUDENTS,
            student_pretrained=ST_STUDENT_PRETRAINED,
            epochs=ST_EPOCHS,
            batch_size=ST_BATCH_SIZE,
            lr=ST_LR,
            weight_decay=ST_WEIGHT_DECAY,
            seed=SEED,
            device=DEVICE,
            topk_percent=ST_TOPK_PERCENT,
        )

    test_dir = Path(PATH) / cls / "test"
    test_paths = list_images(test_dir)
    if len(test_paths) == 0:
        raise RuntimeError(f"No test images found in: {test_dir}")

    test_ds = ImagePathDataset(test_paths, transform=eval_tf)
    test_loader = DataLoader(
        test_ds,
        batch_size=ST_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

    model_st.eval()
    with torch.no_grad():
        for batch in test_loader:
            images = batch["image"].to(DEVICE, non_blocking=True)
            paths = batch["path"]

            anomaly_maps = compute_student_teacher_maps(
                model=model_st,
                images=images,
                image_size=IMAGE_SIZE,
            ).detach().cpu().numpy()

            for path, amap in zip(paths, anomaly_maps):
                submission_rows.append(
                    {
                        "ID": Path(path).stem,
                        "Label": float_matrix_to_q8rle(normalize_map(amap)),
                    }
                )

    dt = time.time() - t0
    print(f"✓ Class {cls} done in {dt:.1f}s — {len(test_paths)} test images")

    del model_st
    del test_ds
    del test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

out_dir = Path("./submission")
out_dir.mkdir(parents=True, exist_ok=True)
sub_df = pd.DataFrame(submission_rows)
sub_df = sub_df.sort_values("ID").reset_index(drop=True)

csv_path = out_dir / "submission_student_teacher_allclasses.csv"
sub_df.to_csv(csv_path, index=False)

zip_path = out_dir / "submission_student_teacher_allclasses.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(csv_path, arcname=csv_path.name)

print("\n" + "=" * 60)
print("Saved submission CSV:", csv_path)
print("Saved ZIP:", zip_path)
print("Total time: %.1f s" % (time.time() - start_all))
print("=" * 60)
